# Final Image-Only Wear Analysis (Sets 1-17)

This notebook trains and evaluates the regression model across all 17 sets, experimenting with feature extraction and target scaling.

In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, ConcatDataset
from torchvision import transforms as T
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from PIL import Image
from copy import deepcopy

import sys
sys.path.append('../src')
from data.image_dataset import WearImageDataset
from models.image_only_model import ImageOnlyWearModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert torch.cuda.is_available(), "CUDA unavailable — STOP."
device = torch.device("cuda")

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))


PyTorch: 2.13.0+cu130
CUDA: 13.0
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
VRAM GB: 7.96


In [2]:
# Dataset Definition
# TRAIN (70%): Sets 1-8, 12-15
# VAL (18%): Sets 9, 10, 16
# TEST (12%): Sets 11, 17

train_sets = [1, 2, 3, 4, 5, 6, 7, 8, 12, 13, 14, 15]
val_sets = [9, 10, 16]
test_sets = [11, 17]

base_dir = Path("../data")
labels_path = base_dir / "raw" / "MATWI" / "labels.csv"
extracted_dir = base_dir / "extracted"

df = pd.read_csv(labels_path)
df = df[df['wear'].notna() & df['ImageName'].notna()]

# Create dataset objects (assuming unscaled initially, we will scale the targets inside the training loop manually)
train_transform = T.Compose([
    T.CenterCrop(2000),
    T.Resize((384, 384)),
    T.RandomRotation(5),
    T.ColorJitter(brightness=0.1, contrast=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

eval_transform = T.Compose([
    T.CenterCrop(2000),
    T.Resize((384, 384)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def create_dataset_for_sets(set_nums, transform):
    datasets = []
    for s in set_nums:
        set_df = df[df['Set'] == s]
        if len(set_df) > 0:
            set_id = f"Set{s}"
            # Check where images are extracted
            img_dir = extracted_dir / set_id
            if not img_dir.exists():
                img_dir = extracted_dir / "MATWI" / set_id / set_id / "images"
                if not img_dir.exists():
                     img_dir = extracted_dir / "MATWI" / set_id
            
            if img_dir.exists():
                ds = WearImageDataset(set_df, base_dir, transform=transform)
                datasets.append(ds)
    if not datasets:
        return None
    return ConcatDataset(datasets)

train_dataset = create_dataset_for_sets(train_sets, train_transform)
val_dataset = create_dataset_for_sets(val_sets, eval_transform)
test_dataset = create_dataset_for_sets(test_sets, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=0)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")


Train samples: 1112
Val samples: 303
Test samples: 248


In [3]:
# Target Scaling (StandardScaler)
# Fit ONLY on the Train split targets to prevent leakage.
train_targets = []
for i in range(len(train_dataset.datasets)):
    ds = train_dataset.datasets[i]
    train_targets.extend(ds.df['wear'].tolist())

train_targets = np.array(train_targets).reshape(-1, 1)

target_scaler = StandardScaler()
target_scaler.fit(train_targets)

print("Target Scaler fitted on Train only.")
print(f"Train Target Mean: {target_scaler.mean_[0]:.2f}")
print(f"Train Target Scale (Std): {target_scaler.scale_[0]:.2f}")


Target Scaler fitted on Train only.
Train Target Mean: 109.05
Train Target Scale (Std): 80.63


In [4]:
def create_model(model_type):
    model = ImageOnlyWearModel()
    if model_type == 'A':
        # End to End Raw B0
        # Everything unfrozen
        for param in model.parameters():
            param.requires_grad = True
    elif model_type == 'B':
        # Frozen Extractor + Head
        for param in model.backbone.parameters():
            param.requires_grad = False
        for param in model.backbone.classifier.parameters():
            param.requires_grad = True
    elif model_type == 'C':
        # Partially unfreeze later layers
        for param in model.backbone.parameters():
            param.requires_grad = False
        # Unfreeze features.7 and features.8 of EfficientNet B0
        for name, param in model.backbone.named_parameters():
            if 'features.7' in name or 'features.8' in name:
                param.requires_grad = True
        for param in model.backbone.classifier.parameters():
            param.requires_grad = True
    
    return model.to(device)

def evaluate(model, loader, scaler):
    model.eval()
    all_preds = []
    all_targets = []
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            # predictions are in scaled space
            preds_scaled = model(imgs).cpu().numpy().reshape(-1, 1)
            
            # inverse transform
            preds = scaler.inverse_transform(preds_scaled).flatten()
            all_preds.extend(preds)
            all_targets.extend(targets.numpy())
            
    all_preds = np.array(all_preds)
    all_targets = np.array(all_targets)
    
    mae = mean_absolute_error(all_targets, all_preds)
    rmse = np.sqrt(mean_squared_error(all_targets, all_preds))
    r2 = r2_score(all_targets, all_preds)
    
    pred_mean, pred_std = np.mean(all_preds), np.std(all_preds)
    pred_min, pred_max = np.min(all_preds), np.max(all_preds)
    
    act_mean, act_std = np.mean(all_targets), np.std(all_targets)
    act_min, act_max = np.min(all_targets), np.max(all_targets)
    
    metrics = {
        'mae': mae, 'rmse': rmse, 'r2': r2,
        'pred_mean': pred_mean, 'pred_std': pred_std, 'pred_min': pred_min, 'pred_max': pred_max,
        'act_mean': act_mean, 'act_std': act_std, 'act_min': act_min, 'act_max': act_max,
        'preds': all_preds, 'targets': all_targets
    }
    return metrics

def train_candidate(model_type, epochs=10):
    print(f"\n--- Training Model {model_type} ---")
    model = create_model(model_type)
    criterion = nn.SmoothL1Loss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
    
    best_val_mae = float('inf')
    best_state = None
    best_metrics = None
    
    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for imgs, targets in train_loader:
            imgs = imgs.to(device)
            
            # Scale target on the fly for loss computation
            targets_scaled = scaler.transform(targets.numpy().reshape(-1, 1))
            targets_scaled = torch.tensor(targets_scaled, dtype=torch.float32).to(device)
            
            optimizer.zero_grad()
            preds_scaled = model(imgs).view(-1, 1)
            loss = criterion(preds_scaled, targets_scaled)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        train_loss /= len(train_loader)
        
        # Evaluate on EXACT SAME validation sets
        val_metrics = evaluate(model, val_loader, scaler=target_scaler)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val MAE: {val_metrics['mae']:.2f} | Val RMSE: {val_metrics['rmse']:.2f} | Val R2: {val_metrics['r2']:.2f}")
        
        if val_metrics['mae'] < best_val_mae:
            best_val_mae = val_metrics['mae']
            best_state = deepcopy(model.state_dict())
            best_metrics = val_metrics
            
    print(f"Best Val MAE for Model {model_type}: {best_val_mae:.2f}")
    
    # Prediction Collapse Check Output for this candidate
    print(f"Prediction Collapse Check [Model {model_type} on Val split]:")
    print(f"  Pred  -> Mean: {best_metrics['pred_mean']:.2f} | Std: {best_metrics['pred_std']:.2f} | Min: {best_metrics['pred_min']:.2f} | Max: {best_metrics['pred_max']:.2f}")
    print(f"  Actual-> Mean: {best_metrics['act_mean']:.2f} | Std: {best_metrics['act_std']:.2f} | Min: {best_metrics['act_min']:.2f} | Max: {best_metrics['act_max']:.2f}")
    
    return best_state, best_metrics


In [5]:
# Train Models A, B, C for max 10 epochs each
scaler = target_scaler

state_A, metrics_A = train_candidate('A', 10)
state_B, metrics_B = train_candidate('B', 10)
state_C, metrics_C = train_candidate('C', 10)

# Select the best model overall based on Validation MAE
candidates = [('A', state_A, metrics_A), ('B', state_B, metrics_B), ('C', state_C, metrics_C)]
best_candidate_name, best_state, best_val_metrics = min(candidates, key=lambda x: x[2]['mae'])

print(f"\nOVERALL BEST MODEL SELECTED: Model {best_candidate_name}")



--- Training Model A ---


Epoch 1/10 | Train Loss: 0.2133 | Val MAE: 44.85 | Val RMSE: 60.41 | Val R2: 0.23


Epoch 2/10 | Train Loss: 0.1503 | Val MAE: 64.50 | Val RMSE: 73.88 | Val R2: -0.16


Epoch 3/10 | Train Loss: 0.0918 | Val MAE: 54.24 | Val RMSE: 69.43 | Val R2: -0.02


Epoch 4/10 | Train Loss: 0.0697 | Val MAE: 55.77 | Val RMSE: 68.40 | Val R2: 0.01


...

Epoch 6/10 | Train Loss: 0.0616 | Val MAE: 54.07 | Val RMSE: 68.85 | Val R2: -0.00


Epoch 7/10 | Train Loss: 0.0669 | Val MAE: 48.21 | Val RMSE: 60.57 | Val R2: 0.22


Epoch 8/10 | Train Loss: 0.0742 | Val MAE: 56.52 | Val RMSE: 71.57 | Val R2: -0.09


Epoch 9/10 | Train Loss: 0.0698 | Val MAE: 55.96 | Val RMSE: 71.77 | Val R2: -0.09


Epoch 10/10 | Train Loss: 0.0613 | Val MAE: 56.60 | Val RMSE: 72.92 | Val R2: -0.13
Best Val MAE for Model C: 43.76
Prediction Collapse Check [Model C on Val split]:
  Pred  -> Mean: 86.42 | Std: 28.16 | Min: 30.14 | Max: 160.02
  Actual-> Mean: 90.64 | Std: 68.70 | Min: 15.00 | Max: 300.00

OVERALL BEST MODEL SELECTED: Model A


In [6]:
# Load Best Model and Evaluate ONCE on Test sets 11, 17
best_model = create_model(best_candidate_name)
best_model.load_state_dict(best_state)

print("\n--- Evaluating Best Model on FINAL TEST SETS ---")
test_metrics = evaluate(best_model, test_loader, scaler)

print(f"Final Test MAE: {test_metrics['mae']:.2f}")
print(f"Final Test RMSE: {test_metrics['rmse']:.2f}")
print(f"Final Test R2: {test_metrics['r2']:.2f}")
print("Final Test Prediction Check:")
print(f"  Pred  -> Mean: {test_metrics['pred_mean']:.2f} | Std: {test_metrics['pred_std']:.2f} | Min: {test_metrics['pred_min']:.2f} | Max: {test_metrics['pred_max']:.2f}")
print(f"  Actual-> Mean: {test_metrics['act_mean']:.2f} | Std: {test_metrics['act_std']:.2f} | Min: {test_metrics['act_min']:.2f} | Max: {test_metrics['act_max']:.2f}")



--- Evaluating Best Model on FINAL TEST SETS ---


Final Test MAE: 45.89
Final Test RMSE: 70.73
Final Test R2: -0.05
Final Test Prediction Check:
  Pred  -> Mean: 110.27 | Std: 21.12 | Min: 62.29 | Max: 161.71
  Actual-> Mean: 134.76 | Std: 69.13 | Min: 30.00 | Max: 750.00


In [7]:
# Material-Wise and Set-Wise Evaluation on Test Sets
# Test sets are 11 (CK45) and 17 (RVS 304)

import csv

test_preds = test_metrics['preds']
test_targets = test_metrics['targets']

# Get Set IDs for test sets
test_set_ids = []
for ds in test_dataset.datasets:
    for i in range(len(ds)):
        test_set_ids.append(ds.df.iloc[i]['Set'])

df_test_results = pd.DataFrame({
    'Set': test_set_ids,
    'Actual': test_targets,
    'Predicted': test_preds
})
df_test_results['Material'] = df_test_results['Set'].apply(lambda x: 'CK45' if x < 12 else 'RVS 304')

print("\nMaterial-wise Metrics:")
for mat in ['CK45', 'RVS 304']:
    sub = df_test_results[df_test_results['Material'] == mat]
    if len(sub) > 0:
        mae = mean_absolute_error(sub['Actual'], sub['Predicted'])
        rmse = np.sqrt(mean_squared_error(sub['Actual'], sub['Predicted']))
        r2 = r2_score(sub['Actual'], sub['Predicted'])
        print(f"{mat} | Samples: {len(sub)} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.2f}")

print("\nSet-wise Metrics:")
per_set_data = []
for s in test_sets:
    sub = df_test_results[df_test_results['Set'] == s]
    if len(sub) > 0:
        mae = mean_absolute_error(sub['Actual'], sub['Predicted'])
        rmse = np.sqrt(mean_squared_error(sub['Actual'], sub['Predicted']))
        r2 = r2_score(sub['Actual'], sub['Predicted'])
        mat = 'CK45' if s < 12 else 'RVS 304'
        print(f"Set {s} ({mat}) | Samples: {len(sub)} | MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.2f}")
        per_set_data.append({'Set': s, 'Material': mat, 'Samples': len(sub), 'MAE': mae, 'RMSE': rmse, 'R2': r2})

Path('../results/image_only').mkdir(parents=True, exist_ok=True)
pd.DataFrame(per_set_data).to_csv('../results/image_only/per_set_metrics.csv', index=False)
df_test_results.to_csv('../results/image_only/predictions.csv', index=False)



Material-wise Metrics:
CK45 | Samples: 101 | MAE: 52.34 | RMSE: 89.72 | R2: -0.26
RVS 304 | Samples: 147 | MAE: 41.46 | RMSE: 53.95 | R2: 0.20

Set-wise Metrics:
Set 11 (CK45) | Samples: 101 | MAE: 52.34 | RMSE: 89.72 | R2: -0.26
Set 17 (RVS 304) | Samples: 147 | MAE: 41.46 | RMSE: 53.95 | R2: 0.20


In [8]:
# Save Artifacts
torch.save(best_state, '../models/image_only_wear/model.pt')

meta = {
    "dataset": "MATWI",
    "sets_used": train_sets + val_sets + test_sets,
    "train_sets": train_sets,
    "val_sets": val_sets,
    "test_sets": test_sets,
    "architecture": f"EfficientNet-B0 (Strategy {best_candidate_name})",
    "preprocessing": "2000px crop -> 384x384 -> ImageNet norm",
    "loss": "SmoothL1Loss",
    "optimizer": "AdamW",
    "learning_rate": 1e-3,
    "target_scaler": "StandardScaler",
    "validation_mae": float(best_val_metrics['mae']),
    "test_mae": float(test_metrics['mae']),
    "test_rmse": float(test_metrics['rmse']),
    "test_r2": float(test_metrics['r2'])
}
with open('../models/image_only_wear/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=4)
    
with open('../results/image_only/metrics.json', 'w') as f:
    json.dump(meta, f, indent=4)

import joblib
joblib.dump(target_scaler, '../models/image_only_wear/target_scaler.pkl')

# Plot Actual vs Predicted
plt.figure(figsize=(8,6))
sns.scatterplot(x=df_test_results['Actual'], y=df_test_results['Predicted'], hue=df_test_results['Material'])
plt.plot([0, 800], [0, 800], 'r--')
plt.xlabel('Actual Wear (um)')
plt.ylabel('Predicted Wear (um)')
plt.title('Final Test: Actual vs Predicted Wear')
plt.savefig('../results/image_only/actual_vs_predicted.png')
plt.close()

# Residuals
residuals = df_test_results['Predicted'] - df_test_results['Actual']
plt.figure(figsize=(8,6))
sns.histplot(residuals, kde=True)
plt.xlabel('Residuals (Predicted - Actual)')
plt.title('Final Test: Residuals Distribution')
plt.savefig('../results/image_only/residuals.png')
plt.close()
